# Continued pretraining with capitalization embeddings

This notebook runs masked-language-model continued pretraining with two heads: the normal BERT token head and a 3-way capitalization head. The default dataset is small enough for a first Colab run. Swap in larger preserved-case corpora after the smoke path is stable.

In [ ]:
from pathlib import Path
import os

RUNPOD_REPO = Path("/workspace/repos/CapitalizationEmbeddings")
COLAB_REPO = Path("/content/drive/MyDrive/Github/CapitalizationEmbeddings")
try:
    from google.colab import drive

    if not COLAB_REPO.exists():
        drive.mount("/content/drive")
except Exception:
    pass

if RUNPOD_REPO.exists():
    os.chdir(RUNPOD_REPO)
elif COLAB_REPO.exists():
    os.chdir(COLAB_REPO)

print("repo:", Path.cwd())
%pip install -q -e . -r requirements-colab.txt

from capitalization_embeddings import configure_huggingface_cache
HF_CACHE_DIR = configure_huggingface_cache()
print("HF cache:", HF_CACHE_DIR)


In [ ]:
from pathlib import Path

if not Path("pyproject.toml").exists():
    raise RuntimeError("Run this notebook from the CapitalizationEmbeddings repo root.")

try:
    from google.colab import drive

    drive.mount("/content/drive")
except Exception:
    pass

In [ ]:
MODEL_NAME = "bert-base-uncased"

# Small default for iteration. Good larger candidates later: Salesforce/wikitext wikitext-103-raw-v1, cc_news, wikimedia/wikipedia, C4 realnewslike.
DATASET_NAME = "Salesforce/wikitext"
DATASET_CONFIG = "wikitext-2-raw-v1"
TEXT_COLUMN = "text"
TRAIN_SPLIT = "train[:5%]"
EVAL_SPLIT = "validation[:10%]"

MAX_LENGTH = 128
MLM_PROBABILITY = 0.15
MAX_STEPS = 1_000
PER_DEVICE_BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 2
LEARNING_RATE = 5e-5

from capitalization_embeddings import checkpoint_dir

OUTPUT_DIR = checkpoint_dir("mlm")

In [ ]:
from collections import Counter

from datasets import load_dataset
from transformers import AutoTokenizer

from capitalization_embeddings import tokenize_with_capitalization

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

raw_train = load_dataset(DATASET_NAME, DATASET_CONFIG, split=TRAIN_SPLIT)
raw_eval = load_dataset(DATASET_NAME, DATASET_CONFIG, split=EVAL_SPLIT)

def has_text(row):
    value = row[TEXT_COLUMN]
    return value is not None and value.strip() != ""

raw_train = raw_train.filter(has_text)
raw_eval = raw_eval.filter(has_text)

def preprocess_batch(examples):
    return tokenize_with_capitalization(
        tokenizer,
        examples[TEXT_COLUMN],
        truncation=True,
        max_length=MAX_LENGTH,
    )

tokenized_train = raw_train.map(
    preprocess_batch,
    batched=True,
    remove_columns=raw_train.column_names,
)
tokenized_eval = raw_eval.map(
    preprocess_batch,
    batched=True,
    remove_columns=raw_eval.column_names,
)

cap_counts = Counter()
sample_size = min(1_000, len(tokenized_train))
for cap_ids in tokenized_train.select(range(sample_size))["capitalization_ids"]:
    cap_counts.update(cap_ids)

print("train rows:", len(tokenized_train))
print("eval rows:", len(tokenized_eval))
print("capitalization id sample counts:", dict(cap_counts))

In [ ]:
import os
import torch
from transformers.trainer_utils import get_last_checkpoint

from capitalization_embeddings import (
    CapitalizedBertForMaskedLM,
    DataCollatorForCapitalizedLanguageModeling,
    make_trainer,
    make_training_arguments,
)

model = CapitalizedBertForMaskedLM.from_uncased_pretrained(MODEL_NAME)
data_collator = DataCollatorForCapitalizedLanguageModeling(
    tokenizer=tokenizer,
    mlm_probability=MLM_PROBABILITY,
)

training_args = make_training_arguments(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=False,
    max_steps=MAX_STEPS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_ratio=0.06,
    logging_steps=25,
    eval_steps=250,
    save_steps=250,
    save_total_limit=3,
    eval_strategy="steps",
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = make_trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
    processing_class=tokenizer,
)

last_checkpoint = get_last_checkpoint(OUTPUT_DIR) if os.path.isdir(OUTPUT_DIR) else None
trainer.train(resume_from_checkpoint=last_checkpoint)
trainer.save_model(f"{OUTPUT_DIR}/final")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final")